In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "bueno2020effects")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Bueno_Guerra2020_pt1Data_NBG_Indirect reputation.csv")
complete_path_2 = os.path.join(original_data_pathway, "Bueno_Guerra2020_pt2Data_NBG_Indirect reputation.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
df1 = pd.read_csv(complete_path_1)
df2 = pd.read_csv(complete_path_2)
df2.rename(columns={"Subject": "ape",
"Condition":"condition_temp",
"Rearing":"rearing_temp"}, inplace=True)

# print(df2.columns)
fulldf = pd.concat([df1,df2], axis = 1)
fulldf.columns = map(str.lower, fulldf.columns)
fulldf=fulldf.applymap(lambda s: s.lower() if type(s) == str else s)

fulldf = fulldf.assign(study_id='bueno2020effects')
fulldf = fulldf.assign(experiment='1')
fulldf = fulldf.assign(experiment_name='prosocial_and_antisocial_choices_and_emotional_reactions')

# df1['experiment']="prosocial_and_antisocial_choices"
# df2['experiment']="emotional_reactions"

In [3]:
fulldf['ape'] = fulldf['ape'].str.rstrip()

comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)

for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
fulldf= fulldf.merge(apedf,left_on='ape', right_on='name', how='left')

fulldf=fulldf.rename(columns={"sex_y": "sex"})


In [4]:
temp = []
for entry in fulldf["condition"]:
    if entry == 2:
        entry = "phase 2_participants_chose_between_prosocial_plus_1_and_antisocial_plus_4"
    elif entry == 3:
        entry = "phase 3_participants_chose_between_prosocial_plus_2_and_antisocial_plus_2"
    temp.append(entry)
fulldf = fulldf.assign(temp_col=temp)
fulldf=fulldf.rename(columns={'temp_col': 'condition_codes'})

In [5]:
# "first choice in phase3": "first choice in phase3_0=antisocial_1=prosocial"
temp = []
for entry in fulldf["first choice in phase3"]:
    if entry == 0:
        entry = "antisocial"
    elif entry == 1:
        entry = "prosocial"
    temp.append(entry)
fulldf = fulldf.assign(temp_col=temp)
fulldf=fulldf.rename(columns={'temp_col': 'first choice in phase3_codes'})


In [6]:
# "rearing": "rearing_0=peer group_1=half or full nursery"
temp = []
for entry in fulldf["rearing"]:
    if entry == 0:
        entry = "peer_group"
    elif entry == 1:
        entry = "half_or_full_nursery"
    temp.append(entry)
# print(fulldf['rearing'])
fulldf = fulldf.assign(temp_col=temp)
fulldf=fulldf.rename(columns={'temp_col': 'rearing_codes'})



In [7]:
# "event": "event_1=fight_2=consolation"
temp = []
for entry in fulldf["event"]:
    if entry == 1:
        entry = "fight"
    elif entry == 2:
        entry = "consolation"
    temp.append(entry)
fulldf = fulldf.assign(temp_col=temp)
fulldf=fulldf.rename(columns={'temp_col': 'event_codes',
    "ape":"participant",
    "age":"age_in_years",
    "first choice in phase3_codes":"first_choice_in_phase3_codes"})




In [8]:
fulldf=fulldf[['study_id',  'experiment_name','participant',
        'age_in_years', 'sex', 'species','condition', 'condition_codes',
         'rearing_codes','choice',
       'first choice in phase3', 'first_choice_in_phase3_codes','event','event_codes',
       'behaviors', 'vocalizations']]

fulldf=fulldf[['study_id',  'experiment_name','participant',
        'age_in_years', 'sex', 'species', 'condition_codes',
        'choice',
        'first_choice_in_phase3_codes','event_codes',
       'behaviors', 'vocalizations']].copy()
fulldf.columns =fulldf.columns.str.replace('_codes', '')

In [9]:
exp1 = fulldf[fulldf['experiment_name'] == 'prosocial_and_antisocial_choices_and_emotional_reactions']
exp1 = exp1.dropna(axis=1, how='all')
# exp2 = fulldf[fulldf['experiment'] == 'emotional_reactions']
# exp2 = exp2.dropna(axis=1, how='all')

In [10]:
comp_out_path_stand_1 = os.path.join(out_pathway, 'bueno2020effects_standardized.csv')
exp1.to_csv(comp_out_path_stand_1, encoding='utf-8-sig', index=False)

# comp_out_path_stand_2 = os.path.join(out_pathway, 'bueno2020effects_exp2_standardized.csv')
# exp2.to_csv(comp_out_path_stand_2, encoding='utf-8-sig', index=False)

In [11]:
names = exp1.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
exp1_glossary=df[["column_name", "description"]]


comp_out_path_glossary = os.path.join(out_pathway, 'bueno2020effects_glossary.csv')
exp1_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
